<!-- 학습 보강 셀 -->

# 10. FAISS 학습 흐름

이 노트북은 LlamaIndex의 인덱싱 흐름에 FAISS 벡터 스토어를 연결하는 예제입니다.
FAISS는 대량의 벡터를 빠르게 검색하기 위한 라이브러리이며, 로컬 실습과 고성능 검색 실험에 자주 사용됩니다.

In [ ]:
# FAISS 벡터 스토어 예제에 필요한 패키지 설치
# - faiss-cpu: 로컬 CPU 기반 벡터 검색 라이브러리
# - llama-index-vector-stores-faiss: LlamaIndex와 FAISS를 연결하는 패키지
# !pip install faiss-cpu llama-index-vector-stores-faiss llama-index-llms-ollama llama-index-embeddings-ollama

In [ ]:
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import StorageContext, VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import load_index_from_storage
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [ ]:
# LLM과 임베딩 모델 설정
llm = Ollama(
    model='gemma2:2b',
    temperature=0,
    request_timeout=120,
)

embed_model = OllamaEmbedding(
    model_name='nomic-embed-text',
)

In [ ]:
# 데이터 로드
# - FAISS에 저장할 원본 문서를 먼저 Document 목록으로 읽습니다.
documents = SimpleDirectoryReader('../NewData/pdf_sample2/').load_data()
print('읽어온 문서 수:', len(documents))

In [ ]:
# FAISS 인덱스 생성
# - nomic-embed-text의 임베딩 차원은 768입니다.
# - 임베딩 모델을 바꾸면 dimension도 해당 모델 차원에 맞춰 바꿔야 합니다.
dimension = 768
faiss_index = faiss.IndexFlatL2(dimension)

<!-- 학습 보강 셀 -->

## dimension이 중요한 이유

FAISS 인덱스의 차원 수는 임베딩 모델이 출력하는 벡터 차원과 반드시 같아야 합니다.
`nomic-embed-text`는 768차원 벡터를 만들기 때문에 `dimension = 768`로 설정합니다.
임베딩 모델을 바꾸면 이 값도 함께 확인해야 합니다.

In [ ]:
# FAISS를 LlamaIndex의 인덱싱 및 검색 파이프라인에 통합합니다.
vector_store = FaissVectorStore(
    faiss_index=faiss_index,
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

<!-- 학습 보강 셀 -->

## FAISS와 LlamaIndex의 역할 분담

FAISS는 벡터를 빠르게 찾는 검색 엔진 역할을 하고, LlamaIndex는 문서 로딩, Node 관리, 쿼리 엔진 연결을 담당합니다.
둘을 함께 쓰면 벡터 검색 성능과 RAG 파이프라인 편의성을 동시에 얻을 수 있습니다.

In [ ]:
# 인덱스 생성 및 데이터 임베딩
# - 생성된 벡터는 위에서 만든 FAISS vector_store에 저장됩니다.
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
)

In [ ]:
# 인덱스 저장
# - FAISS 파일과 LlamaIndex 메타데이터를 함께 저장해야 나중에 정상 로드할 수 있습니다.
persist_dir = './faiss_storage'
index.storage_context.persist(persist_dir=persist_dir)

<!-- 학습 보강 셀 -->

## FAISS 저장소를 다룰 때 주의할 점

FAISS 저장 파일은 확장자가 `.json`처럼 보여도 내부가 바이너리일 수 있습니다.
따라서 사람이 직접 수정하지 말고, LlamaIndex와 FAISS API를 통해 저장하고 로드하는 것이 안전합니다.

In [ ]:
# 저장된 FAISS 벡터 스토어 로드
# - 변수명 오타를 방지하기 위해 loaded_vector_store로 통일합니다.
loaded_vector_store = FaissVectorStore.from_persist_dir(persist_dir)

In [ ]:
# 저장된 인덱스 로드 위치 정의 및 인덱스 복원
# - vector_store와 persist_dir을 모두 넘겨야 벡터와 문서 메타데이터를 함께 읽습니다.
loaded_storage_context = StorageContext.from_defaults(
    vector_store=loaded_vector_store,
    persist_dir=persist_dir,
)

loaded_index = load_index_from_storage(
    loaded_storage_context,
    embed_model=embed_model,
)

<!-- 학습 보강 셀 -->

## 복원 검증 방법

저장소에서 복원한 인덱스가 제대로 동작하는지는 같은 질문을 메모리 인덱스와 복원 인덱스에 모두 던져 비교하면 됩니다.
답변이 완전히 같지 않을 수는 있지만, 참조하는 문서와 핵심 내용은 일관되어야 합니다.

---
### 메모리 인덱스 질의

In [ ]:
# 메모리에 있는 원본 인덱스로 쿼리 엔진 생성
query_engine = index.as_query_engine(llm=llm)

In [ ]:
# 메모리 인덱스에 질의 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘'
response = query_engine.query(query)

print()
print('질문:', query)
print('답변:', response)

### 파일에서 불러온 FAISS 인덱스 질의

In [ ]:
# 저장소에서 복원한 인덱스로 쿼리 엔진 생성
loaded_query_engine = loaded_index.as_query_engine(llm=llm)

In [ ]:
# 복원된 인덱스에 같은 질문 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘'
response = loaded_query_engine.query(query)

print()
print('질문:', query)
print('답변:', response)